https://www.kaggle.com/competitions/titanic/

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data_path = "./data/train.csv"

In [4]:
df = pd.read_csv(data_path)

In [5]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Some initial observations on the data:

- `Passengerid` should not contain salient information
- `Survived` is the label
- `Pclass` ordinal feature
- `Name` should not contain salient information
- `Sex` categorical feature
- `Age` continuous feature
- `SibSp` # of siblings / spouses aboard the Titanic. Ordinal feature
- `Parch` # of parents / children aboard the Titanic. Ordinal feature
- `Ticket` categorical feature. There may be some information hidden in the ticket, but it is probably captured by `Fare` and `Cabin`
- `Fare` continuous feature. Fare may be relevant. For example, higher fare may be correlated to richer person who had an higher chance to survive
- `Cabin` categorical feature. Location of the cabins may be relevant for the survival. However, the spacial location of the cabins cannot be directly understood from the numbers alone and additional information is necessary
- `Embarked` categorical feature. It may or may not be relevant. For example, Port of Embarkation may be correlated to the position of the cabins and represent an easier to treat compared to the cabin number. It could be correlated to the social status of the people and so to the fare.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

`Age`, `Cabin` and `Embarked` miss data

In [20]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Check whether categorical features have more values than expected due to typo, synonyms

In [30]:
for col in df.columns:
    print(df[col].value_counts())
    print('\n\n')

PassengerId
1      1
2      1
3      1
4      1
5      1
      ..
887    1
888    1
889    1
890    1
891    1
Name: count, Length: 891, dtype: int64



Survived
0    549
1    342
Name: count, dtype: int64



Pclass
3    491
1    216
2    184
Name: count, dtype: int64



Name
Braund, Mr. Owen Harris                                1
Cumings, Mrs. John Bradley (Florence Briggs Thayer)    1
Heikkinen, Miss. Laina                                 1
Futrelle, Mrs. Jacques Heath (Lily May Peel)           1
Allen, Mr. William Henry                               1
                                                      ..
Montvila, Rev. Juozas                                  1
Graham, Miss. Margaret Edith                           1
Johnston, Miss. Catherine Helen "Carrie"               1
Behr, Mr. Karl Howell                                  1
Dooley, Mr. Patrick                                    1
Name: count, Length: 891, dtype: int64



Sex
male      577
female    314
Name: count, dtype: in

Notes:

- Multiple people have the same `Ticket` and `Cabin`
- Some values for `Cabin` seems to be composed by several cabin numbers, e.g. C23 C25 C27